# 07 — Embedding providers

memtomem supports multiple embedding backends.  This notebook compares
three of them side-by-side:

1. **BM25-only** (`provider="none"`) — keyword search, zero external dependencies
2. **ONNX** (`provider="onnx"`) — local dense search via fastembed, no server needed
3. **Ollama** (`provider="ollama"`) — local dense search via Ollama server

The first two run entirely offline with no GPU; Ollama requires a running
server.  You can run the notebook with only BM25 + ONNX if Ollama is not
available — the Ollama cells are guarded.

## Prerequisites

```bash
uv pip install "memtomem[onnx]" jupyter ipykernel
# Ollama section is optional — install memtomem[ollama] and run 'ollama serve'
```

## Shared setup: temp directory and corpus

In [ ]:
import json
import os
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

corpus = {
    "rollback.md": "# Rollback drills\n\nAlways practice rolling back a deployment in staging before you actually need to do it in production.\n",
    "migrations.md": "# Database migrations\n\nRun migrations in a dry-run mode first. Dry runs exercise the migration path without touching production rows.\n",
    "monitoring.md": "# Deployment monitoring\n\nWatch the error rate and p99 latency dashboards for fifteen minutes after every deploy.\n",
    "playbook.md": "# On-call deploy playbook\n\nWhen a deploy fails, follow the deploy playbook: page the on-call, start the rollback, write a post-incident note.\n",
}

for name, content in corpus.items():
    (notes_dir / name).write_text(content, encoding="utf-8")

print(f"Corpus: {len(corpus)} files in {notes_dir}")

In [ ]:
import memtomem.config as _cfg
from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import Components, create_components, close_components

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None

QUERY = "rollback after failed deploy"


def pretty(title, results):
    print(f"--- {title} ---")
    for r in results:
        name = Path(r.chunk.metadata.source_file).name
        snippet = r.chunk.content[:80].replace("\n", " ")
        print(f"  [{r.rank}] {name:16s} score={r.score:.3f} via={r.source:<5} — {snippet}...")
    print()

## Provider 1: BM25-only (`provider="none"`)

Zero dependencies — pure keyword search using SQLite FTS5.  This is the
default after `mm init` (quick start option).

In [ ]:
db_bm25 = tmp_path / "bm25.db"

config = Mem2MemConfig()
config.storage.sqlite_path = db_bm25
config.indexing.memory_dirs = [notes_dir]
config.embedding.provider = "none"  # BM25-only
config.embedding.model = ""
config.embedding.dimension = 0

comp_bm25: Components = await create_components(config)
await comp_bm25.index_engine.index_path(notes_dir)

results, stats = await comp_bm25.search_pipeline.search(query=QUERY, top_k=4)
pretty(f"BM25-only (bm25={stats.bm25_candidates}, dense={stats.dense_candidates})", results)

await close_components(comp_bm25)

## Provider 2: ONNX (`provider="onnx"`)

Local dense search via [fastembed](https://github.com/qdrant/fastembed).
The model (`all-MiniLM-L6-v2`, ~22 MB) is downloaded automatically on
first use and cached in `~/.cache/fastembed/`.

No server, no GPU, no Docker — just `pip install memtomem[onnx]`.

In [ ]:
db_onnx = tmp_path / "onnx.db"

config = Mem2MemConfig()
config.storage.sqlite_path = db_onnx
config.indexing.memory_dirs = [notes_dir]
config.embedding.provider = "onnx"
config.embedding.model = "all-MiniLM-L6-v2"  # 384d, ~22 MB
config.embedding.dimension = 384

comp_onnx: Components = await create_components(config)
await comp_onnx.index_engine.index_path(notes_dir)

results, stats = await comp_onnx.search_pipeline.search(query=QUERY, top_k=4)
pretty(f"ONNX (bm25={stats.bm25_candidates}, dense={stats.dense_candidates})", results)

await close_components(comp_onnx)

## Provider 3: Ollama (optional)

If Ollama is running, this cell shows the same search with
`nomic-embed-text` (768d).  Skipped automatically if Ollama is not
reachable.

In [ ]:
import httpx

_ollama_ok = False
try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
    _ollama_ok = True
except Exception:
    print("Ollama not reachable — skipping this section.")
    print("Start Ollama and pull nomic-embed-text to enable it.")

if _ollama_ok:
    db_ollama = tmp_path / "ollama.db"

    config = Mem2MemConfig()
    config.storage.sqlite_path = db_ollama
    config.indexing.memory_dirs = [notes_dir]
    config.embedding.provider = "ollama"
    config.embedding.model = "nomic-embed-text"
    config.embedding.dimension = 768

    comp_ollama: Components = await create_components(config)
    await comp_ollama.index_engine.index_path(notes_dir)

    results, stats = await comp_ollama.search_pipeline.search(query=QUERY, top_k=4)
    pretty(f"Ollama (bm25={stats.bm25_candidates}, dense={stats.dense_candidates})", results)

    await close_components(comp_ollama)

## Switching models

The ONNX provider supports three models out of the box:

| Model | Dimension | Size | Best for |
|-------|-----------|------|----------|
| `all-MiniLM-L6-v2` | 384 | ~22 MB | Lightweight English |
| `bge-small-en-v1.5` | 384 | ~33 MB | Better English accuracy |
| `bge-m3` | 1024 | ~1.2 GB | Multilingual (KR/EN/JP/CN) |

To switch, change `config.embedding.model` and `config.embedding.dimension`,
then re-index.  If you already have an index, use `mm embedding-reset`
to handle the dimension migration safely.

## Cleanup

In [ ]:
_cfg.load_config_overrides = _orig_loader
tmp.cleanup()
print("clean.")